# Introduction to SAOS

Welcome to the **Solar Adaptive Optics Simulator (SAOS)** tutorial! 
In this notebook, we will introduce the basic concepts of setting up an open-loop propagation with a point source.

SAOS builds on an object-oriented architecture where every element of the AO loop is represented by a class (`Telescope`, `Atmosphere`, `Source`, `LightPath`, etc.).

In [ ]:
import logging
import matplotlib.pyplot as plt
import numpy as np

from SAOS.LoggingHelper import LoggingHelper
from SAOS.Telescope import Telescope
from SAOS.Atmosphere import Atmosphere
from SAOS.Source import Source
from SAOS.LightPath import LightPath

# Set up the logger
logger_helper = LoggingHelper(logging.INFO)
logger = logger_helper.logger

## 1. Defining the Telescope
First, we define the parameters of our telescope (e.g., European Solar Telescope or EST). We must specify the primary mirror diameter, the central obstruction, the phase screen resolution, and the simulation sampling time.

In [ ]:
diameter = 4.149       # Primary mirror diameter [m]
obs_diameter = 1.3     # Central obstruction [m]
sampling_time = 1/2000 # Sampling time [s] (2 kHz)
resolution = 216       # Resolution of the phase screen [px]
tel_fov = 60           # Telescope Field of View [arcsec]

est_tel = Telescope(diameter=diameter,
                    resolution=resolution,
                    centralObstruction=obs_diameter/diameter,
                    samplingTime=sampling_time,
                    fov=tel_fov,
                    logger=logger)

## 2. Setting up the Atmosphere
The multi-layer atmosphere requires specifying $r_0$, $L_0$, and the profile across multiple altitudes, including fractional $r_0$ weights, wind directions, and speeds.

In [ ]:
atm = Atmosphere(r0=0.21,
                 L0=25,
                 fractionalR0=[0.53, 0.37, 0.05, 0.03, 0.02],
                 altitude=[100, 1500, 5000, 10000, 15000],
                 windDirection=[45, 90, 180, 67, 2],
                 windSpeed=[6.7, 8.5, 12.7, 23.4, 8.3],
                 telescope=est_tel,
                 zenith=0,
                 logger=logger)

atm.initializeAtmosphere()

## 3. Light Source and Propagation Path
We create a standard `Source` (point source) or an `ExtendedSource` (like the Sun). 
Then, we group everything using the `LightPath` abstraction, which controls the order of phase addition.

In [ ]:
# A point source (Natural Guide Star)
ngs = Source(magnitude=5,
             optBand='R4',
             coordinates=[0,0],
             logger=logger)

# Create the light path and initialize it with our objects
path = LightPath(logger)
path.initialize_path(src=ngs, atm=atm, tel=est_tel)

# Update the atmosphere and propagate the light through the path
atm.update()
path.propagate()

print("Propagation successful! The phase at the telescope pupil is available in path.src.phase")

## 4. Visualizing the Phase
Finally, we can visualize the incoming phase screen.

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(path.src.phase.numpy() * est_tel.pupil, cmap='jet')
plt.colorbar(label='Phase [rad]')
plt.title('Turbulent Phase at Telescope Pupil')
plt.show()